# Notebook 02: Cross-Validation for Honest Evaluation

## Learning Objectives

By the end of this notebook you will be able to:

1. Use `NeuralForecast.cross_validation()` to evaluate models across multiple time windows
2. Compare NHITS against a DLinear baseline with a fair evaluation protocol
3. Compute and visualize error distributions to understand forecast stability
4. Explain how `n_windows`, `step_size`, `val_size`, and `test_size` affect evaluation

**Estimated time**: 12–15 minutes (including ~8 min training)

**Prerequisites**: Notebook 01 (Training NHITS), Guide 02 (Hyperparameter Tuning)

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

print("Imports complete")

## 1. Load Data

Same French Bakery dataset as Notebook 01 — all products in Nixtla long format.

In [ ]:
# Load the French Bakery dataset
url = (
    "https://raw.githubusercontent.com/Nixtla/transfer-learning-time-series"
    "/main/datasets/french_bakery.csv"
)
df = pd.read_csv(url, parse_dates=["ds"])

products = df["unique_id"].unique().tolist()
print(f"Products: {products}")
print(f"Date range: {df['ds'].min().date()} to {df['ds'].max().date()}")
print(f"Rows: {len(df)}")

## 2. Why Single Train/Test Splits Are Unreliable

In Notebook 01 we used a single train/test split: train on everything up to the last 7 days, test on the final 7 days. This has a critical weakness: the test window might be unusually easy or unusually hard.

The diagram below shows how rolling window cross-validation solves this:

```
Window 1:  [────────── train ──────────] [test: week 1]
Window 2:  [─────────── train ───────────] [test: week 2]
Window 3:  [──────────── train ────────────] [test: week 3]
Window 4:  [───────────── train ─────────────] [test: week 4]
```

We get one MAE per window. If accuracy varies wildly across windows, the model is unstable.

In [ ]:
# Visualize the cross-validation window structure
# This is a conceptual diagram — not running the model yet
H = 7   # forecast horizon
N_WINDOWS = 4
STEP_SIZE = H  # advance one horizon between windows

max_date = df["ds"].max()
min_date = df["ds"].min()
total_days = (max_date - min_date).days

print(f"Forecast horizon H = {H} days")
print(f"Number of evaluation windows: {N_WINDOWS}")
print(f"Step size between windows: {STEP_SIZE} days")
print()

# Show the cutoff date for each window
print("Window structure:")
for i in range(N_WINDOWS, 0, -1):
    cutoff = max_date - pd.Timedelta(days=i * STEP_SIZE)
    test_start = cutoff + pd.Timedelta(days=1)
    test_end = cutoff + pd.Timedelta(days=H)
    print(f"  Window {N_WINDOWS - i + 1}: train ends {cutoff.date()}  |  test {test_start.date()} to {test_end.date()}")

## 3. Define Both Models

We compare two models side by side:

- **DLinear**: a linear baseline (trend decomposition + linear projection). Simple, fast, and surprisingly strong.
- **NHITS**: our neural forecaster with multi-rate sampling and hierarchical interpolation.

If NHITS cannot beat DLinear, the signal in the data is likely linear — neural complexity is not justified.

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, DLinear
from neuralforecast.losses.pytorch import MAE

# DLinear: linear baseline
# Fast to train, strong on datasets with dominant linear trends
model_baseline = DLinear(
    h=H,
    input_size=28,
    max_steps=500,
)

# NHITS: neural model with multi-rate sampling
# scaler_type='robust' handles bakery sales outliers
# MAE() loss is appropriate for count data with occasional spikes
model_nhits = NHITS(
    h=H,
    input_size=28,
    max_steps=500,
    scaler_type="robust",
    loss=MAE(),
)

# NeuralForecast trains both models in a single .fit() call
nf = NeuralForecast(models=[model_baseline, model_nhits], freq="D")

print("Models configured:")
print(f"  - DLinear (baseline): linear decomposition")
print(f"  - NHITS: hierarchical multi-rate neural model")

## 4. Run Cross-Validation

`cross_validation()` trains the models and evaluates them on `n_windows` rolling test windows. The output DataFrame has one row per (series, forecast date, window).

Key output columns:
- `cutoff`: the last date in the training data for that window
- `y`: the actual observed value
- `DLinear`: DLinear's prediction
- `NHITS`: NHITS's prediction

In [ ]:
print(f"Running {N_WINDOWS}-window cross-validation...")
print("(This trains models and evaluates on 4 rolling windows — ~8 min on CPU)")
print()

# cross_validation trains once and evaluates on all n_windows test sets
# step_size=H means each window advances exactly one forecast horizon
cv_df = nf.cross_validation(
    df=df,
    n_windows=N_WINDOWS,
    step_size=H,
)

print("Cross-validation complete.")
print(f"\nOutput shape: {cv_df.shape}")
print(f"Columns: {cv_df.columns.tolist()}")
print(f"\nFirst 5 rows:")
cv_df.head()

## 5. Aggregate Metrics

Compute MAE and MSE across all products and windows, then break down by window to check stability.

In [ ]:
from utilsforecast.losses import mae, mse

# Overall MAE and MSE for each model
mae_baseline = mae(cv_df["y"], cv_df["DLinear"]).mean()
mae_nhits = mae(cv_df["y"], cv_df["NHITS"]).mean()

mse_baseline = mse(cv_df["y"], cv_df["DLinear"]).mean()
mse_nhits = mse(cv_df["y"], cv_df["NHITS"]).mean()

improvement_pct = (mae_baseline - mae_nhits) / mae_baseline * 100

print("Overall cross-validation metrics (all products, all windows):")
print(f"{'Model':<12} {'MAE':>8} {'MSE':>10} {'RMSE':>8}")
print("-" * 42)
print(f"{'DLinear':<12} {mae_baseline:>8.2f} {mse_baseline:>10.2f} {mse_baseline**0.5:>8.2f}")
print(f"{'NHITS':<12} {mae_nhits:>8.2f} {mse_nhits:>10.2f} {mse_nhits**0.5:>8.2f}")
print(f"\nNHITS improvement over DLinear: {improvement_pct:+.1f}%")

if improvement_pct > 5:
    print("NHITS clearly outperforms the linear baseline — neural modeling is justified.")
elif improvement_pct > 0:
    print("NHITS slightly better — consider tuning input_size or max_steps further.")
else:
    print("DLinear is competitive — the signal may be predominantly linear.")

## 6. Metrics by Window

Checking per-window accuracy reveals whether the models degrade at specific times of year. A model that performs well in February but poorly in December has a seasonal blind spot.

In [ ]:
# Compute MAE per window for each model
# The 'cutoff' column identifies which window each row belongs to
per_window = []
for cutoff_date in sorted(cv_df["cutoff"].unique()):
    window_df = cv_df[cv_df["cutoff"] == cutoff_date]
    per_window.append({
        "cutoff": cutoff_date.date(),
        "DLinear_MAE": mae(window_df["y"], window_df["DLinear"]).mean(),
        "NHITS_MAE": mae(window_df["y"], window_df["NHITS"]).mean(),
    })

per_window_df = pd.DataFrame(per_window)
per_window_df["NHITS_improvement_%"] = (
    (per_window_df["DLinear_MAE"] - per_window_df["NHITS_MAE"])
    / per_window_df["DLinear_MAE"] * 100
).round(1)

print("Per-window MAE:")
print(per_window_df.to_string(index=False, float_format="{:.2f}".format))

## 7. Error Distribution Visualization

Box plots of absolute errors show the full error distribution — not just the mean. A model with a lower mean but higher variance might be less useful in practice than one with slightly higher mean but tight error distribution.

A well-calibrated model has:
- Symmetric error distribution centered near zero
- Few outlier errors (short whiskers)
- Similar performance across products

In [ ]:
# Compute absolute errors for both models
cv_df["abs_error_DLinear"] = (cv_df["y"] - cv_df["DLinear"]).abs()
cv_df["abs_error_NHITS"] = (cv_df["y"] - cv_df["NHITS"]).abs()
cv_df["residual_DLinear"] = cv_df["y"] - cv_df["DLinear"]
cv_df["residual_NHITS"] = cv_df["y"] - cv_df["NHITS"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Absolute error distributions (box plots per product)
ax = axes[0]
data_to_plot = []
labels = []
for product in products:
    prod_df = cv_df[cv_df["unique_id"] == product]
    data_to_plot.append(prod_df["abs_error_DLinear"].values)
    labels.append(f"{product}\nDLinear")
    data_to_plot.append(prod_df["abs_error_NHITS"].values)
    labels.append(f"{product}\nNHITS")

bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
# Color alternating boxes: DLinear=steel blue, NHITS=tomato
for i, patch in enumerate(bp["boxes"]):
    patch.set_facecolor("steelblue" if i % 2 == 0 else "tomato")
    patch.set_alpha(0.6)

ax.set_title("Absolute Error Distribution by Product", fontsize=11)
ax.set_ylabel("Absolute Error")
ax.tick_params(axis="x", labelsize=8)

# Plot 2: MAE by window, both models
ax = axes[1]
x = range(len(per_window_df))
width = 0.35
bars1 = ax.bar(
    [xi - width / 2 for xi in x], per_window_df["DLinear_MAE"],
    width=width, label="DLinear", color="steelblue", alpha=0.7, edgecolor="black"
)
bars2 = ax.bar(
    [xi + width / 2 for xi in x], per_window_df["NHITS_MAE"],
    width=width, label="NHITS", color="tomato", alpha=0.7, edgecolor="black"
)
ax.set_xticks(x)
ax.set_xticklabels(
    [f"Window {i+1}\n(cutoff {per_window_df['cutoff'].iloc[i]})" for i in x],
    fontsize=8
)
ax.set_title("MAE by Evaluation Window", fontsize=11)
ax.set_ylabel("MAE")
ax.legend()

fig.suptitle("Cross-Validation Error Analysis: DLinear vs NHITS", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Residual Analysis: Is the Model Biased?

Residuals = actual - predicted.

- A well-calibrated model has residuals centered at zero (no systematic bias)
- Positive residual: model under-predicted (conservative)
- Negative residual: model over-predicted (optimistic)

For inventory management, consistent under-prediction leads to stockouts. Consistent over-prediction leads to waste. Both are costly for a bakery.

In [ ]:
# Compare residual distributions for NHITS
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram of residuals for all products combined
ax = axes[0]
ax.hist(
    cv_df["residual_DLinear"], bins=30, alpha=0.6,
    color="steelblue", label="DLinear", edgecolor="white"
)
ax.hist(
    cv_df["residual_NHITS"], bins=30, alpha=0.6,
    color="tomato", label="NHITS", edgecolor="white"
)
ax.axvline(0, color="black", linewidth=1.5, linestyle="--", label="Zero")
ax.set_xlabel("Residual (actual - predicted)")
ax.set_ylabel("Count")
ax.set_title("Residual Distribution (all products)")
ax.legend()

# Bias by product
ax = axes[1]
bias_data = []
for product in products:
    prod_df = cv_df[cv_df["unique_id"] == product]
    bias_data.append({
        "product": product,
        "DLinear bias": prod_df["residual_DLinear"].mean(),
        "NHITS bias": prod_df["residual_NHITS"].mean(),
    })

bias_df = pd.DataFrame(bias_data).set_index("product")
bias_df.plot(kind="bar", ax=ax, color=["steelblue", "tomato"], alpha=0.7, edgecolor="black")
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Mean Residual by Product\n(closer to 0 = less bias)")
ax.set_ylabel("Mean Residual")
ax.tick_params(axis="x", rotation=15)
ax.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

# Print numerical summary
print("\nBias summary (mean residual per product):")
print(bias_df.round(2).to_string())

## 9. val_size and test_size Explained

When calling `nf.fit()` directly, you can control how the model handles end-of-series data:

- `val_size`: the last `val_size` time steps are used for **internal validation** (learning rate scheduling, early stopping). They are **not** used for gradient updates.
- `test_size`: the last `test_size` steps are excluded from training entirely and not used for internal validation either.

These parameters are useful when you want to compare multiple model configurations with `fit()` rather than `cross_validation()`.

In [ ]:
# Demonstrate val_size and test_size
# val_size=14: hold out last 14 days for validation during training
# test_size=7: exclude last 7 days from training (our evaluation set)

print("Demonstrating val_size and test_size parameters:")
print()

series_length = df[df["unique_id"] == products[0]]["ds"].nunique()
val_size = 14
test_size = H

print(f"Total time steps per series: {series_length}")
print(f"  val_size={val_size}: last {val_size} steps used for validation (not in gradient updates)")
print(f"  test_size={test_size}: last {test_size} steps excluded entirely")
print(f"  Effective training steps: {series_length - val_size - test_size}")
print()

# Fit with val_size to show the API
# (This retrains — commented out to save time in demo mode)
# nf_with_val = NeuralForecast(
#     models=[NHITS(h=H, input_size=28, max_steps=500, scaler_type='robust')],
#     freq='D'
# )
# nf_with_val.fit(df, val_size=14)
# forecast_with_val = nf_with_val.predict()

print("Typical usage:")
print("  nf.fit(df, val_size=14)           # validates during training")
print("  nf.fit(df, val_size=14, test_size=7)  # val + hold-out test")
print("  nf.cross_validation(df, n_windows=4)  # rolling eval (recommended)")
print()
print("Recommendation: use cross_validation() for model comparison.")
print("Use val_size in fit() only when training the final production model.")

## 10. Summary

In this notebook you:

1. Ran rolling window cross-validation with `nf.cross_validation()` on the French Bakery dataset
2. Compared NHITS against a DLinear baseline using MAE and MSE
3. Visualized error distributions — box plots and per-window bar charts
4. Analyzed residuals to check for systematic bias
5. Learned the difference between `val_size`, `test_size`, and `cross_validation()`

### Key Result

If NHITS outperformed DLinear (positive improvement %), the bakery data contains non-linear patterns that the neural architecture can exploit. The hierarchical decomposition across multiple frequency bands is what gives NHITS this advantage.

If the improvement was small, the bakery sales signal is largely linear (strong weekly seasonality dominates) — consider whether the added training cost of NHITS is justified for your use case.

### Next Steps

- **Exercise 01**: Try different `input_size` values and compare MAE across configurations
- **Module 02**: Probabilistic forecasting — replace point forecasts with prediction intervals
- **Guide 02**: Tune `scaler_type` and loss function to squeeze more performance from NHITS